In [16]:
import os

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.init as init

import numpy as np

import albumentations as A
from collections import defaultdict

import SimpleITK
import SimpleITK as sitk

import re
import random

In [17]:
BASE_DIR = "database/"  # Name of the dataset folder

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():  # mac
    device = torch.device("mps")
else:
    device = torch.device("cpu")


print("Using device:", device)

sitk.ProcessObject_SetGlobalWarningDisplay(False)

Using device: mps


## 1. FCN1:


In [18]:
def get_file_paths(directory):
    """
    Get all file paths in a directory and its subdirectories.
    """
    file_paths = []  # creates an empty list
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_paths.append(os.path.join(root, file))
    return file_paths


# p: probability of applying the transformations
train_transform = A.Compose(
    [
        A.Resize(
            height=256, width=256, p=1
        ),  # re-sized all the images to 256 × 256 pixels
        A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-10, 10), p=1),
    ]
)

resize = A.Compose([A.Resize(height=256, width=256, p=1)])


### 1.1 FCN1 - Dataset


In [19]:
class FCN1Dataset(Dataset):
    def __init__(self, filepath, transform=None):
        self.slices = []
        self.transform = transform

        for path in filepath:
            img = self._sitk_load(path, normalize=True)

            mask_path = path.replace(".nii.gz", "_gt.nii.gz")
            mask = self._sitk_load(mask_path, normalize=False)
            mask = (mask == 3).astype(np.float32)

            # regex to find info
            patient = path.split("/")[-2]
            frame = int(re.search(r"frame(\d+)", path).group(1))

            for i in range(img.shape[0]):
                self.slices.append(
                    {
                        "image": img[i],
                        "mask": mask[i],
                        "patient": patient,
                        "frame": frame,
                        "slice_idx": i,
                    }
                )

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        item = self.slices[idx]
        img = item["image"]
        mask = item["mask"]

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]

        img = torch.from_numpy(img).float()
        mask = torch.from_numpy(mask).float()

        img = img.unsqueeze(0)
        mask = mask.unsqueeze(0)

        metadata = {
            "patient": item["patient"],
            "frame": item["frame"],
            "slice_idx": item["slice_idx"],
        }

        return img, mask, metadata

    def _sitk_load(self, filepath, normalize=False) -> np.ndarray:
        """Loads an image using SimpleITK and returns the image and its metadata.
        Args:
            filepath: Path to the image.

        Returns:
            - ([N], H, W), Image array.

        """
        # Load image and save info
        image = SimpleITK.ReadImage(str(filepath))
        # Extract numpy array from the SimpleITK image object
        im_array = np.squeeze(SimpleITK.GetArrayFromImage(image))
        if normalize:
            im_array = im_array.astype(np.float32)
            im_array = (im_array - im_array.mean()) / (im_array.std() + 1e-8)

        return im_array


In [20]:
all_files = get_file_paths("database/")

img_paths = [
    file
    for file in all_files
    if file.endswith(".nii.gz") and "_gt" not in file and "_4d" not in file
]
train_val_paths = [file for file in img_paths if "training" in file]


# Choose 10 random patients from pattient 001 to patiento 100 for the validation set
random.seed(42)
patients = list(range(1, 101))  # patient001 - patient100
val_patients = random.sample(patients, 10)

train_paths = []
validation_paths = []

# Extracts the patient number from the file path and checks whether that patient belongs to the validation set.
for f in train_val_paths:
    patient = int(f.split("patient")[1][:3])
    if patient in val_patients:
        validation_paths.append(f)
    else:
        train_paths.append(f)

# ------

ds_train = FCN1Dataset(train_paths, train_transform)
ds_validation = FCN1Dataset(validation_paths, resize)

test_image_paths = [file for file in img_paths if "testing" in file]
test_image_sorted = sorted(
    test_image_paths, key=lambda x: int(re.search(r"patient(\d+)", x).group(1))
)

### 1.2. FCN1 - Model Architecture


In [21]:
class EncoderFCN1(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        skip = x
        x = self.pool(x)
        return x, skip


class DecoderFCN1(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Input: 512 - Output: 256
        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        # Input: 256 *2 -> (because of skip connections)
        self.conv1 = nn.Conv2d(out_channels * 2, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x, skip):
        x = self.upconv(x)
        # dim = 1: Concatenate along the channel dimension.
        x = torch.cat((x, skip), dim=1)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x


class FCN1Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder1 = EncoderFCN1(1, 32)
        self.encoder2 = EncoderFCN1(32, 64)
        self.encoder3 = EncoderFCN1(64, 128)
        self.encoder4 = EncoderFCN1(128, 256)

        # Bottleneck
        self.bottleneck1 = nn.Conv2d(256, 512, 3, padding=1)
        self.bn_b1 = nn.BatchNorm2d(512)
        # bottleneck2
        self.bottleneck2 = nn.Conv2d(512, 512, 3, padding=1)
        self.bn_b2 = nn.BatchNorm2d(512)
        self.relu = nn.ReLU()

        # Decoder
        self.decode1 = DecoderFCN1(512, 256)
        self.decode2 = DecoderFCN1(256, 128)
        self.decode3 = DecoderFCN1(128, 64)
        self.decode4 = DecoderFCN1(64, 32)

        self.final_conv = nn.Conv2d(32, 1, 3, padding=1)

        # init weights AFTER all layers exist
        self.apply(self.init_weights)

    def forward(self, x):
        x, s1 = self.encoder1(x)
        x, s2 = self.encoder2(x)
        x, s3 = self.encoder3(x)
        x, s4 = self.encoder4(x)

        x = self.bottleneck1(x)
        x = self.bn_b1(x)
        x = self.relu(x)
        x = self.bottleneck2(x)
        x = self.bn_b2(x)
        x = self.relu(x)

        x = self.decode1(x, s4)
        x = self.decode2(x, s3)
        x = self.decode3(x, s2)
        x = self.decode4(x, s1)

        x = self.final_conv(x)
        return x

    def init_weights(self, m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
            init.kaiming_normal_(m.weight, nonlinearity="relu")

### 1.3. FCN1 - Grid Search & Training


### 1.4. Grid Search Execution


In [22]:
def calculate_centroid(mask_np):
    """
    Calculates the centroid (y, x) of a binary mask.
    Returns (y_mean, x_mean) or None if the mask is empty.
    """
    y_indices, x_indices = np.where(mask_np > 0)
    if len(y_indices) == 0:
        return None
    return (np.mean(y_indices), np.mean(x_indices))


def fill_missing_centroids(data):
    groups = defaultdict(list)
    for item in data:
        group_key = (str(item["patient"]), str(item["frame"]))
        groups[group_key].append(item)

    result = []

    for group_key, group in groups.items():
        group.sort(key=lambda x: x["slice_idx"])

        # Forward Fill para pred_centroid y gt_centroid
        for i in range(1, len(group)):
            if group[i]["pred_centroid"] is None:
                group[i]["pred_centroid"] = group[i - 1]["pred_centroid"]
            if group[i]["gt_centroid"] is None:
                group[i]["gt_centroid"] = group[i - 1]["gt_centroid"]

        # Backward Fill para pred_centroid y gt_centroid
        for i in range(len(group) - 2, -1, -1):
            if group[i]["pred_centroid"] is None:
                group[i]["pred_centroid"] = group[i + 1]["pred_centroid"]
            if group[i]["gt_centroid"] is None:
                group[i]["gt_centroid"] = group[i + 1]["gt_centroid"]

        result.extend(group)
    return result


def calculate_centroid_metrics(data):
    errors = []

    for item in data:
        pred = item["pred_centroid"]
        gt = item["gt_centroid"]

        if pred is not None and gt is not None:
            pred_arr = np.array(pred)
            gt_arr = np.array(gt)

            # np.linalg.norm is a function in the Python NumPy library that calculates the norm of a vector or matrix.
            error = np.linalg.norm(pred_arr - gt_arr)
            errors.append(error)

    if not errors:
        return None, None, None, None
    mean_error = np.mean(errors)
    min_error = np.min(errors)
    max_error = np.max(errors)
    std_error = np.std(errors)

    return mean_error, min_error, max_error, std_error


def evaluate_fcn1(model, test_loader):
    model.eval()
    results = []

    with torch.no_grad():
        for img_batch, mask_batch, metadata_batch in test_loader:
            img_batch = img_batch.to(device)
            mask_batch = mask_batch.to(device)
            pred_batch = (torch.sigmoid(model(img_batch)) > 0.5).float()

            for i in range(img_batch.size(0)):
                results.append(
                    {
                        "patient": str(metadata_batch["patient"][i]),
                        "frame": str(metadata_batch["frame"][i]),
                        "slice_idx": int(metadata_batch["slice_idx"][i]),
                        "pred_centroid": calculate_centroid(
                            pred_batch[i].squeeze().cpu().numpy()
                        ),
                        "gt_centroid": calculate_centroid(
                            mask_batch[i].squeeze().cpu().numpy()
                        ),
                    }
                )
    results.sort(key=lambda x: (x["patient"], x["frame"], x["slice_idx"]))

    result = fill_missing_centroids(results)
    mean_error, min_error, max_error, std_error = calculate_centroid_metrics(result)

    return mean_error, min_error, max_error, std_error


In [23]:
def train_model(
    model,
    train_loader,
    validation_loader,
    criterion=nn.BCEWithLogitsLoss(),
    epochs=20,
    learning_rate=0.001,
):
    """
    Train the model on the training set and validate it on the validation set.

    Return the training and validation metrics.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    metrics = {
        "train_loss": [],
        "val_loss": [],
    }

    for epoch in range(epochs):
        losses = 0
        model.train()
        for img, mask, _ in train_loader:
            img = img.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()
            output = model(img)
            loss = criterion(output, mask)
            loss.backward()
            optimizer.step()
            losses += loss.item()

        metrics["train_loss"].append(losses / len(train_loader))
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for img, mask, _ in validation_loader:
                img = img.to(device)
                mask = mask.to(device)

                output = model(img)
                val_loss += criterion(output, mask).item()
            metrics["val_loss"].append(val_loss / len(validation_loader))

    return metrics


In [24]:
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)


N_EPOCHS = 100


def grid_search():
    batch_sizes = [8, 16, 32]
    lr_values = [0.01, 0.001, 0.0001]

    best_params = {}
    best_metric = float("inf")
    for batch_size in batch_sizes:
        for lr_ in lr_values:
            torch.manual_seed(42)
            np.random.seed(42)

            model = FCN1Net().to(device)
            criterion = nn.BCEWithLogitsLoss()
            generator = (
                torch.Generator(device="cuda") if device.type == "cuda" else None
            )

            dataloader_args = {
                "batch_size": batch_size,
                "generator": generator,
            }

            dl_train = DataLoader(ds_train, **dataloader_args, shuffle=True)
            dl_val = DataLoader(ds_validation, **dataloader_args, shuffle=False)

            print("Training...")
            metrics = train_model(
                model,
                dl_train,
                dl_val,
                criterion=criterion,
                epochs=N_EPOCHS,
                learning_rate=lr_,
            )
            final_train_loss = metrics["train_loss"][-1]
            final_val_loss = metrics["val_loss"][-1]

            print(f"Evaluating model: batch_size={batch_size}, lr_={lr_}, ")
            # 2. Evaluate
            mean_error, min_error, max_error, std_error = evaluate_fcn1(model, dl_val)

            # 3.Compare the results and choose the best hyperparameters
            if mean_error is not None:
                print(
                    f"batch_size={batch_size}, lr_={lr_}, "
                    f"train_loss={final_train_loss:.5f}, val_loss={final_val_loss:.5f}, "
                    f"mean_error={mean_error:.4f}, max_error={max_error:.4f}, "
                    f"min_error={min_error:.4f}, std_error={std_error:.4f}"
                )

                if mean_error < best_metric:
                    best_metric = mean_error
                    best_params = {
                        "batch_size": batch_size,
                        "lr_": lr_,
                        "final_train_loss": final_train_loss,
                        "final_val_loss": final_val_loss,
                        "mean_error": mean_error,
                        "std_error": std_error,
                        "min_error": min_error,
                        "max_error": max_error,
                    }

            elif mean_error is None:
                print("Could not compute the centroid metric for this configuration.")

    return best_params


best_params = grid_search()
print("\nGrid search finished")
print(f"Best distance ( = {(best_params.get('mean_error', -1))}")

print(f"Best batch_size = {best_params.get('batch_size', -1)}")
print(f"Best lr_ = {best_params.get('lr_', -1)}")


Training...
Evaluating model: batch_size=8, lr_=0.01, 
batch_size=8, lr_=0.01, train_loss=0.00171, val_loss=0.00337, mean_error=2.6406, max_error=106.0842, min_error=0.0589, std_error=11.5132
Training...
Evaluating model: batch_size=8, lr_=0.001, 
batch_size=8, lr_=0.001, train_loss=0.00160, val_loss=0.00339, mean_error=0.8422, max_error=8.4162, min_error=0.0150, std_error=1.2062
Training...
Evaluating model: batch_size=8, lr_=0.0001, 
batch_size=8, lr_=0.0001, train_loss=0.00131, val_loss=0.00400, mean_error=0.9994, max_error=17.8661, min_error=0.0466, std_error=1.7519
Training...
Evaluating model: batch_size=16, lr_=0.01, 
batch_size=16, lr_=0.01, train_loss=0.00190, val_loss=0.00326, mean_error=2.1867, max_error=54.5359, min_error=0.0129, std_error=7.0845
Training...
Evaluating model: batch_size=16, lr_=0.001, 
batch_size=16, lr_=0.001, train_loss=0.00157, val_loss=0.00388, mean_error=1.4535, max_error=30.5979, min_error=0.0271, std_error=3.8507
Training...
Evaluating model: batch_s

<i>
<h3>Saved Results</h3>
Training...

Evaluating model: batch*size=8, lr*=0.01,

batch*size=8, lr*=0.01, train_loss=0.00171, val_loss=0.00337, mean_error=2.6406, max_error=106.0842, min_error=0.0589, std_error=11.5132

Training...

Evaluating model: batch*size=8, lr*=0.001,

batch*size=8, lr*=0.001, train_loss=0.00160, val_loss=0.00339, mean_error=0.8422, max_error=8.4162, min_error=0.0150, std_error=1.2062

Training...

Evaluating model: batch*size=8, lr*=0.0001,

batch*size=8, lr*=0.0001, train_loss=0.00131, val_loss=0.00400, mean_error=0.9994, max_error=17.8661, min_error=0.0466, std_error=1.7519

Training...

Evaluating model: batch*size=16, lr*=0.01,

batch*size=16, lr*=0.01, train_loss=0.00190, val_loss=0.00326, mean_error=2.1867, max_error=54.5359, min_error=0.0129, std_error=7.0845

Training...

Evaluating model: batch*size=16, lr*=0.001,

batch*size=16, lr*=0.001, train_loss=0.00157, val_loss=0.00388, mean_error=1.4535, max_error=30.5979, min_error=0.0271, std_error=3.8507

Training...

Evaluating model: batch*size=16, lr*=0.0001,

batch*size=16, lr*=0.0001, train_loss=0.00133, val_loss=0.00456, mean_error=1.4421, max_error=49.8535, min_error=0.0305, std_error=3.9454

Training...

Evaluating model: batch*size=32, lr*=0.01,

batch*size=32, lr*=0.01, train_loss=0.00179, val_loss=0.00378, mean_error=1.8913, max_error=102.2262, min_error=0.0293, std_error=8.5818

Training...

Evaluating model: batch*size=32, lr*=0.001,

batch*size=32, lr*=0.001, train_loss=0.00258, val_loss=0.01734, mean_error=17.5953, max_error=128.5933, min_error=0.0755, std_error=22.3369

Training...

...

Grid search finished

Best distance ( = 0.8421865113973013

Best batch_size = 8

Best lr\_ = 0.001

</i>
